# 01 Â· Real hotel bookings: B2B vs direct, and a revenue bridge

**REAL DATASET** - Antonio, Almeida & Nunes (2019), *Hotel booking demand datasets*, Data in Brief, CC BY 4.0. Two hotels in Portugal, 2015-2017, used to demonstrate the analysis, **not ETG data**.

The functions live in `src/analysis_real.py`; this notebook runs them step by step and shows the results.

In [1]:
import sys, warnings
warnings.filterwarnings("ignore")   # keep outputs clean
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))   # reuse the project's scripts
import pandas as pd
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)
from common import OUT, SIM, RAW, PUBLIC
import analysis_real as ar
d = ar.load()
d.shape

(119390, 36)

## 1. Data quality first
No booking ID in the file: identical rows may be real group bookings, so they are kept and flagged. Rows with a zero or negative rate, zero nights or zero guests are excluded from price statistics only.

In [2]:
ar.dq_checks(d)

,check,rows,decision
0,Rows in file,119390,"Matches the 119,390 bookings described in the ..."
1,Fully duplicated rows,31994,No booking ID in the file: identical rows can ...
2,Negative ADR,1,Excluded from price statistics
3,"ADR above 1,000 EUR",1,"Outlier, excluded from price statistics"
4,"Zero ADR, not complimentary",1279,"Kept in volumes, excluded from price statistics"
5,Zero nights,715,Day use or error: excluded from nights and ADR
6,Zero guests,180,Probable data-entry error: excluded
7,Missing country,488,Shown as 'Unknown'
8,Undefined channel,5,Grouped as 'Other'


## 2. Channels: price, cancellations, lead time
TA/TO = travel agents and tour operators (the B2B channel). ADR = room revenue / room nights on stays that were not cancelled.

In [3]:
ar.kpis_by(d, 'channel').round(3)

,channel,bookings,cancel_rate,median_lead_time_days,avg_nights,room_nights_stayed,revenue_eur,adr_eur,share_of_bookings,share_of_revenue
4,TA/TO,97870,0.410,83.0,3.537,204026,20802010.34,101.958,0.820,0.800
1,Direct,14645,0.175,19.0,3.194,35934,4323529.31,120.319,0.123,0.166
0,Corporate,6677,0.221,9.0,2.381,11789,827864.92,70.224,0.056,0.032
2,GDS,193,0.192,10.0,1.959,261,33007.96,126.467,0.002,0.001
3,Other,5,0.800,2.0,3.400,5,563.50,112.700,0.000,0.000


In [4]:
ar.kpis_by(d, 'market_segment').round(3)

,market_segment,bookings,cancel_rate,median_lead_time_days,avg_nights,room_nights_stayed,revenue_eur,adr_eur,share_of_bookings,share_of_revenue
6,Online TA,56477,0.367,58.0,3.574,118498.0,13707139.09,115.674,0.473,0.527
5,Offline TA/TO,24219,0.343,109.0,3.904,67423.0,5659291.94,83.937,0.203,0.218
4,Groups,19811,0.611,162.0,2.993,24250.0,1869129.52,77.078,0.166,0.072
3,Direct,12606,0.153,16.0,3.206,32553.0,4097842.40,125.882,0.106,0.158
2,Corporate,5295,0.187,7.0,2.093,8454.0,577892.19,68.357,0.044,0.022
1,Complementary,743,0.131,3.0,1.647,145.0,4812.53,33.190,0.006,0.000
0,Aviation,237,0.219,3.0,3.608,692.0,70868.36,102.411,0.002,0.003
7,Undefined,2,1.000,1.5,1.500,NaN,NaN,NaN,0.000,NaN


A suspicious pattern to check with the data owner before drawing any conclusion: non-refundable bookings cancel almost always.

In [5]:
d.groupby('deposit_type')['is_canceled'].agg(['size', 'mean']).round(3)

,size,mean
deposit_type,,
No Deposit,104641,0.284
Non Refund,14587,0.994
Refundable,162,0.222


## 3. Concentration on travel agents (Pareto)

In [6]:
p = ar.agent_pareto(d)
print(f"{len(p)} agents; top 1 = {p.cum_share.iloc[0]:.0%}, top 10 = {p.cum_share.iloc[9]:.0%}, "
      f"agents needed for 80% = {(p.cum_share < 0.8).sum() + 1}")
p.head(10)

313 agents; top 1 = 30%, top 10 = 72%, agents needed for 80% = 18


,agent,revenue,rank,cum_share
0,Agent 9,6990224.55,1,0.299953
1,Agent 240,3596513.26,2,0.454280
2,Agent 250,1406148.90,3,0.514619
3,Agent 14,1161724.24,4,0.564469
4,Agent 7,978087.77,5,0.606439
5,Agent 40,657565.64,6,0.634655
6,Agent 241,640851.19,7,0.662154
7,Agent 6,510401.09,8,0.684056
8,Agent 28,420434.74,9,0.702097
9,Agent 8,414317.71,10,0.719875


## 4. Revenue bridge, August -> September 2016
Revenue = sum over segments of room nights x ADR.

* **Volume** = (N1 - N0) x ADR0 average
* **Mix** = N1 x sum((w1 - w0) x (ADR0_s - ADR0 average))
* **Price** = N1 x sum(w1 x (ADR1_s - ADR0_s))

The three effects add up exactly to the change (check = 0).

In [7]:
bridge, drill = ar.revenue_bridge(d, '2016-08', '2016-09', ['hotel', 'market_segment'])
print('check:', round(bridge.attrs['check'], 6))
bridge

check: 0.0


,step,eur
0,Revenue 2016-08,1.807654e+06
1,Volume,-9.323521e+04
2,Mix,-2.188515e+05
3,Price,-2.066201e+05
4,Revenue 2016-09,1.288947e+06


In [8]:
drill.groupby('hotel')[['volume', 'mix', 'price', 'total_change']].sum().round(0)

,volume,mix,price,total_change
hotel,,,,
City Hotel,-55737.0,-47610.0,58576.0,-35723.0
Resort Hotel,-37498.0,-171241.0,-265196.0,-482984.0


In [9]:
drill.sort_values('total_change')[['hotel', 'market_segment', 'N0', 'N1', 'P0', 'P1', 'volume', 'mix', 'price', 'total_change']].round(0).head(8)

,hotel,market_segment,N0,N1,P0,P1,volume,mix,price,total_change
12,Resort Hotel,Online TA,2596.0,1510.0,201.0,125.0,-158456.0,-52367.0,-114643.0,-332831.0
9,Resort Hotel,Direct,1213.0,693.0,212.0,136.0,-75872.0,-30431.0,-53315.0,-163780.0
6,City Hotel,Online TA,4620.0,3309.0,125.0,134.0,-191285.0,22430.0,28606.0,-135266.0
11,Resort Hotel,Offline TA/TO,1845.0,2089.0,120.0,76.0,35602.0,-8744.0,-93202.0,-63891.0
3,City Hotel,Direct,782.0,435.0,140.0,143.0,-50630.0,1716.0,1278.0,-47410.0
7,Resort Hotel,Complementary,2.0,0.0,2.0,2.0,-292.0,273.0,0.0,-4.0
1,City Hotel,Complementary,0.0,8.0,68.0,68.0,1167.0,-619.0,0.0,548.0
0,City Hotel,Aviation,0.0,38.0,108.0,108.0,5545.0,-1425.0,0.0,4120.0


**Reading:** the resort hotel explains almost all the drop. Room nights barely fell, because a group block refilled the resort at a low rate while online-agency and direct guests left after the summer peak: mix and price, not volume.